In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")

import anthropic
client = anthropic.Anthropic()

import pandas as pd


In [ ]:


load_dotenv("../.env")
client = anthropic.Anthropic()

MODEL = "claude-haiku-4-5-20251001"

tos_out = pd.read_csv("../results/tos_test_outputs.csv")
probe_out = pd.read_csv("../results/probe_outputs.csv")

In [ ]:
SYSTEM_PROMPT = """
You summarize legal texts (terms of service and contract clauses) in plain English for an ordinary reader in a style resembeling a tldr
Follow these Rules:
1: Write one or two short sentences.
2: Only write what the text actually says. Do not add anything that is not in the text.
3: Use simple, everyday words.
4: Reply with the summary only."""


In [ ]:



EXAMPLES = [
    (
        "apple administers and operates the www apple com site from its location in cupertino california usa other apple sites may be administered and operated from various locations outside the united states. although the site is accessible worldwide not all features products or services discussed referenced provided or offered through or on the site are available to all persons or in all geographic locations or appropriate or available for use outside the united states. apple reserves the right to limit in its sole discretion the provision and quantity of any feature product or service to any person or geographic area. any offer for any feature product or service made on the site is void where prohibited. if you choose to access the site from outside the united states you do so on your own initiative and you are solely responsible for complying with applicable local laws.",
        "not all apple products or services are available to everyone or in all locations. apple may limit any product or service to any person or area. products and services are not available where prohibited by law. you are responsible for complying with local laws if accessing the website outside of the u s.",
    ),
    (
        "you re free to stop using our services at any time. we also reserve the right to suspend or end the services at any time at our discretion and without notice. for example we may suspend or terminate your use of the services if you re not complying with these terms or use the services in a manner that would cause us legal liability disrupt the services or disrupt others use of the services. except for paid accounts we reserve the right to terminate and delete your account if you haven t accessed our services for 12 consecutive months. we ll of course provide you with notice via the email address associated with your account before we do so.",
        "dropbox can end their services at any time without notice. for free accounts they can delete your account if you haven t used dropbox for 12 months. they ll email you before doing so.",
    ),
    (
        "billing. you can increase your storage space and add paid features to your account turning your account into a paid account. we ll automatically bill you from the date you convert to a paid account and on each periodic renewal until cancellation. you re responsible for all applicable taxes and we ll charge tax when required to do so. no refunds. you may cancel your dropbox paid account at any time but you won t be issued a refund. downgrades. your paid account will remain in effect until it s cancelled or terminated under these terms. if you don t pay for your paid account on time we reserve the right to suspend it or reduce your storage to free space levels. you re responsible for all taxes and we may charge taxes when required to do so. changes. we may change the fees in effect but will give you advance notice of these changes via a message to the email address associated with your account.",
        "you re responsible to pay any bills you re not entitled to any refunds and if you re late on payments dropbox can suspend or reduce the storage on your account. they can also change their pricing but will email you if they do.",
    ),
]

In [ ]:
def summarize_claude(text):
    messages = []
    for original, summary in EXAMPLES:
        messages.append({"role": "user", "content": original})
        messages.append({"role": "assistant", "content": summary})
    messages.append({"role": "user", "content": text})

    response = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        system=SYSTEM_PROMPT,
        messages=messages,
        extra_body={"temperature": 0},
    )

    if response.stop_reason == "max_tokens":
        print("WARNING: answer was cut off")

    answer = ""
    for block in response.content:
        if block.type == "text":
            answer = answer + block.text
    return answer.strip()

In [ ]:
for text in tos_out["original_text"].head(3):
    print("ORIGINAL:", text)
    print("CLAUDE:  ", summarize_claude(text))
    print()


In [ ]:
tos_haiku = []
for text in tos_out["original_text"]:
    tos_haiku.append(summarize_claude(text))

tos_out["haiku"] = tos_haiku
tos_out[["original_text", "haiku"]].to_csv("../results/tos_test_outputs_haiku.csv", index=False)
print("done")
probe_haiku = []
for text in probe_out["clause_text"]:
    probe_haiku.append(summarize_claude(text))

probe_out["haiku"] = probe_haiku
probe_out[["clause_text", "haiku"]].to_csv("../results/probe_outputs_haiku.csv", index=False)
print("probe done")